# CARSBench — Colab GPU Scale Study

Clean execution notebook for the CARSBench baseline benchmark.

Workflow:
1. Mount Drive
2. Clone/update repo
3. Install package
4. Verify CUDA/GPU
5. Connect persistent result folders
6. Run tests
7. Run acquisition scale study if needed
8. Run final acquisition evaluation
9. Run F-window diagnostics
10. Run chemistry scale study if needed
11. Run final chemistry evaluation
12. Consolidate baseline results
13. Optional Git commit/push using a Colab Secret

**Colab `/content` is temporary. Persist results under `MyDrive/CARSBench_colab/results/`.**


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Clone or update CARSBench

In [ ]:
from pathlib import Path

REPO_DIR = Path("/content/CARSBench")
REPO_URL = "https://github.com/rhouhou/CARSBench.git"

if not REPO_DIR.exists():
    %cd /content
    !git clone {REPO_URL}
else:
    %cd /content/CARSBench
    !git pull --ff-only

%cd /content/CARSBench

## 3. Install CARSBench

In [ ]:
%cd /content/CARSBench
!pip install -e . -q

## 4. Verify GPU / CUDA

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Colab.")

print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 5. Connect persistent result folders

These symlinks are runtime-only. They make experiment results survive Colab restarts. Do not commit them to Git.


In [ ]:
from pathlib import Path
import shutil

REPO_RESULTS = Path("/content/CARSBench/results")
DRIVE_RESULTS = Path("/content/drive/MyDrive/CARSBench_colab/results")

REPO_RESULTS.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

for name in ["scale_study", "final_evaluation", "baseline_summary"]:
    target = DRIVE_RESULTS / name
    link = REPO_RESULTS / name
    target.mkdir(parents=True, exist_ok=True)

    if link.is_symlink():
        link.unlink()
    elif link.exists():
        if link.is_dir():
            shutil.rmtree(link)
        else:
            link.unlink()

    link.symlink_to(target, target_is_directory=True)
    print(f"{link} -> {target}")

## 6. Run tests

In [ ]:
%cd /content/CARSBench
!pytest -q

## 7. Inspect saved results before running anything expensive

If a completed result already exists on Drive, do not retrain it.


In [ ]:
from pathlib import Path

scale_root = Path("/content/CARSBench/results/scale_study")
final_root = Path("/content/CARSBench/results/final_evaluation")

print("Acquisition selection results:",
      len(list((scale_root / "acquisition").glob("**/selection_result.json")))
      if (scale_root / "acquisition").exists() else 0)

print("Chemistry selection results:",
      len(list((scale_root / "chemistry").glob("**/selection_result.json")))
      if (scale_root / "chemistry").exists() else 0)

print("Final acquisition:",
      (final_root / "acquisition" / "final_acquisition_evaluation.json").exists())

print("Final chemistry:",
      (final_root / "chemistry" / "final_chemistry_evaluation.json").exists())

# Acquisition track

Formal source domain: `A_typical`

Training sizes: **5k, 10k, 25k, 50k, 100k**

Model seeds: **101, 202, 303**

OOD data are not used for model selection.


## 8. Optional CUDA verification

In [ ]:
# Run only when verifying a fresh CUDA environment.
# %cd /content/CARSBench
# !python scripts/09_cuda_5k_verification.py

## 9. Acquisition training-size study

In [ ]:
from pathlib import Path
import subprocess

%cd /content/CARSBench

train_sizes = [5_000, 10_000, 25_000, 50_000, 100_000]
model_seeds = [101, 202, 303]
result_root = Path("/content/CARSBench/results/scale_study/acquisition")

for train_size in train_sizes:
    for seed in model_seeds:
        result_file = result_root / f"A_train{train_size}_seed{seed}" / "selection_result.json"

        if result_file.exists():
            print(f"SKIP existing: train={train_size}, seed={seed}")
            continue

        print(f"RUN acquisition: train={train_size}, seed={seed}")
        subprocess.run(
            [
                "python",
                "scripts/08_run_acquisition_scale_study.py",
                "--train-size", str(train_size),
                "--model-seed", str(seed),
                "--device", "cuda",
            ],
            check=True,
        )

## 10. Final acquisition evaluation

In [ ]:
from pathlib import Path

%cd /content/CARSBench

out = Path("/content/CARSBench/results/final_evaluation/acquisition/final_acquisition_evaluation.json")

if out.exists():
    print("Already exists:", out)
else:
    !python scripts/10_evaluate_acquisition_final.py

## 11. F-window mode diagnostic

In [ ]:
from pathlib import Path

%cd /content/CARSBench

out = Path("/content/CARSBench/results/final_evaluation/acquisition/F_window_mode_breakdown.json")

if out.exists():
    print("Already exists:", out)
else:
    !python scripts/11_diagnose_window_modes.py

# Chemistry track

Formal source domain: `H_biochemical_source`

`I_biochemical_target` and `K_joint_ood` are test-only and must not be used for training-size selection.


## 12. Chemistry training-size study

In [ ]:
from pathlib import Path
import subprocess

%cd /content/CARSBench

train_sizes = [5_000, 10_000, 25_000, 50_000, 100_000]
model_seeds = [101, 202, 303]
result_root = Path("/content/CARSBench/results/scale_study/chemistry")

for train_size in train_sizes:
    for seed in model_seeds:
        result_file = result_root / f"H_train{train_size}_seed{seed}" / "selection_result.json"

        if result_file.exists():
            print(f"SKIP existing: train={train_size}, seed={seed}")
            continue

        print(f"RUN chemistry: train={train_size}, seed={seed}")
        subprocess.run(
            [
                "python",
                "scripts/12_run_chemistry_scale_study.py",
                "--train-size", str(train_size),
                "--model-seed", str(seed),
                "--device", "cuda",
            ],
            check=True,
        )

## 13. Final chemistry evaluation

In [ ]:
from pathlib import Path

%cd /content/CARSBench

out = Path("/content/CARSBench/results/final_evaluation/chemistry/final_chemistry_evaluation.json")

if out.exists():
    print("Already exists:", out)
else:
    !python scripts/13_evaluate_chemistry_final.py

## 14. Consolidate baseline results

In [ ]:
%cd /content/CARSBench
!python scripts/14_summarize_baseline_results.py

## 15. Verify consolidated outputs

In [ ]:
from pathlib import Path

summary_root = Path("/content/CARSBench/results/baseline_summary")

for filename in [
    "baseline_summary.json",
    "acquisition_learning_curve.csv",
    "chemistry_learning_curve.csv",
    "acquisition_scale_runs.csv",
    "chemistry_scale_runs.csv",
    "final_acquisition.csv",
    "final_chemistry.csv",
    "F_window_modes.csv",
    "glycogen_484_recovery.csv",
]:
    path = summary_root / filename
    print(f"{filename:38s}", "OK" if path.exists() else "MISSING")

plot_root = summary_root / "plots"
if plot_root.exists():
    print("\nPlots:")
    for path in sorted(plot_root.glob("*.png")):
        print(" ", path.name)

# Optional Git workflow

Use this only when code changed.

Store a fine-grained GitHub token in **Colab Secrets** under `GITHUB_TOKEN`.

Recommended permissions:
- Resource owner: `rhouhou`
- Repository: `CARSBench`
- Contents: **Read and write**
- Metadata: **Read-only**

Never paste the token directly into a notebook cell or commit it.


## 16. Load GitHub token from Colab Secrets

In [ ]:
import os
from google.colab import userdata

os.environ["GH_TOKEN"] = userdata.get("GITHUB_TOKEN")

# Safe check: prints username, not the token.
!gh api user --jq .login

## 17. Restore Git-tracked result paths before committing

Remove runtime Drive symlinks first so Git does not interpret tracked result files as deleted.


In [ ]:
%cd /content/CARSBench

!if [ -L results/scale_study ]; then rm results/scale_study; fi
!if [ -L results/final_evaluation ]; then rm results/final_evaluation; fi
!if [ -L results/baseline_summary ]; then rm results/baseline_summary; fi

!git restore results/scale_study 2>/dev/null || true
!git restore results/final_evaluation 2>/dev/null || true

!git status --short

## 18. Configure Git identity

In [ ]:
%cd /content/CARSBench

GIT_NAME = "Rola Houhou"
GIT_EMAIL = "YOUR_GITHUB_EMAIL_OR_NOREPLY"

!git config user.name "{GIT_NAME}"
!git config user.email "{GIT_EMAIL}"

!git config user.name
!git config user.email

## 19. Commit/push code

Stage only intended source-code files. Do not stage `results/`.


In [ ]:
%cd /content/CARSBench

# Example:
# !git add scripts/14_summarize_baseline_results.py
# !git commit -m "Add consolidated baseline result reporting"

!gh auth setup-git --hostname github.com --force
!git push origin main

# Restart checklist

After a Colab runtime restart:

1. Mount Drive
2. Clone/update repo
3. Install with `pip install -e .`
4. Verify GPU
5. Reconnect persistent result folders
6. Run tests
7. Inspect existing results
8. Only launch runs that are actually missing
